In [1]:
import pandas as pd
import numpy as np
import great_expectations as gx
import json
import warnings
warnings.filterwarnings("ignore")

print(f"Great Expectations version: {gx.__version__}")


Great Expectations version: 1.18.2


In [2]:
df = pd.read_csv("../data/loan_data_clean.csv")

print(f"Data loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nColumns:")
for col in df.columns:
    print(f"  - {col}")

Data loaded: 120,262 rows x 12 columns

Columns:
  - Unnamed: 0
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents


In [3]:
context = gx.get_context()
print("GE context created!")
print(f"Context type: {type(context)}")

GE context created!
Context type: <class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>


In [4]:
data_source = context.data_sources.add_pandas(
    name="loan_data_source"
)

data_asset = data_source.add_dataframe_asset(
    name="loan_data_asset"
)

batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "loan_batch"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Data source created!")
print(f"Batch size: {len(df):,} rows")

Data source created!
Batch size: 120,262 rows


In [5]:
suite = context.suites.add(
    gx.ExpectationSuite(name="loan_data_quality_suite")
)

print("Expectation suite created!")

Expectation suite created!


In [7]:
#  — Target column must exist
suite.add_expectation(
    gx.expectations.ExpectColumnToExist(column="SeriousDlqin2yrs")
)
print("✅ Rule 1: Target column exists")

# Rule 2 — Target must be 0 or 1 only
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="SeriousDlqin2yrs",
        value_set=[0, 1]
    )
)
print("✅ Rule 2: Target values only 0 or 1")

# Rule 3 — Age must not be null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="age")
)
print("✅ Rule 3: Age not null")

# Rule 4 — Age must be between 18 and 100
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="age",
        min_value=18,
        max_value=100
    )
)
print("✅ Rule 4: Age between 18 and 100")

# Rule 5 — Monthly income not null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="MonthlyIncome"
    )
)
print("✅ Rule 5: Monthly income not null")

# Rule 6 — Monthly income must be positive
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="MonthlyIncome",
        min_value=0,
        max_value=None
    )
)
print("✅ Rule 6: Monthly income >= 0")

# Rule 7 — Debt ratio must be between 0 and 50
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="DebtRatio",
        min_value=0,
        max_value=50
    )
)
print("✅ Rule 7: Debt ratio between 0 and 50")

# Rule 8 — Credit utilization not null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="RevolvingUtilizationOfUnsecuredLines"
    )
)
print("✅ Rule 8: Credit utilization not null")

# Rule 9 — At least 100,000 rows
# Rule 9 — At least 100,000 rows
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=100000,
        max_value=None
    )
)
print("✅ Rule 9: At least 100,000 rows")

# Rule 10 — Exactly 11 columns
suite.add_expectation(
    gx.expectations.ExpectTableColumnCountToBeBetween(
        min_value=11,
        max_value=11
    )
)
print("✅ Rule 10: Exactly 11 columns")

print("\nAll 10 expectations added!")

✅ Rule 1: Target column exists
✅ Rule 2: Target values only 0 or 1
✅ Rule 3: Age not null
✅ Rule 4: Age between 18 and 100
✅ Rule 5: Monthly income not null
✅ Rule 6: Monthly income >= 0
✅ Rule 7: Debt ratio between 0 and 50
✅ Rule 8: Credit utilization not null
✅ Rule 9: At least 100,000 rows
✅ Rule 10: Exactly 11 columns

All 10 expectations added!


In [8]:
# Create validation definition
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="loan_data_validation",
        data=batch_definition,
        suite=suite
    )
)

# Run validation
results = validation_definition.run(
    batch_parameters={"dataframe": df}
)

print("="*55)
print("VALIDATION RESULTS")
print("="*55)
print(f"Overall Success: {results.success}")
print(f"\nResults per expectation:")
for result in results.results:
    status = "✅ PASS" if result.success else "❌ FAIL"
    expectation = result.expectation_config.type
    print(f"  {status} — {expectation}")

Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]

VALIDATION RESULTS
Overall Success: False

Results per expectation:
  ✅ PASS — expect_column_to_exist
  ✅ PASS — expect_column_values_to_be_in_set
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_table_row_count_to_be_between
  ❌ FAIL — expect_table_column_count_to_be_between


In [9]:
print(f"Actual column count: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")


Actual column count: 12
Columns: ['Unnamed: 0', 'SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [10]:
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

print(f"Columns now: {df.shape[1]}")
print(df.columns.tolist())

Columns now: 11
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [11]:
# Fix 2 — Update Rule 10 with correct column count
suite.add_expectation(
    gx.expectations.ExpectTableColumnCountToBeBetween(
        min_value=11,
        max_value=11
    )
)
print("✅ Rule 10 updated: exactly 11 columns")

# Fix 3 — Rerun validation on clean dataframe
results = validation_definition.run(
    batch_parameters={"dataframe": df}
)

print(f"\nOverall Success: {results.success}")
print(f"\nResults per expectation:")
for result in results.results:
    status = "✅ PASS" if result.success else "❌ FAIL"
    expectation = result.expectation_config.type
    print(f"  {status} — {expectation}")

✅ Rule 10 updated: exactly 11 columns


Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]


Overall Success: True

Results per expectation:
  ✅ PASS — expect_column_to_exist
  ✅ PASS — expect_column_values_to_be_in_set
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_be_between
  ✅ PASS — expect_column_values_to_not_be_null
  ✅ PASS — expect_table_row_count_to_be_between
  ✅ PASS — expect_table_column_count_to_be_between


In [12]:
passed = sum(1 for r in results.results if r.success)
failed = sum(1 for r in results.results if not r.success)

print("="*55)
print(f"Total expectations : {len(results.results)}")
print(f"Passed             : {passed} ✅")
print(f"Failed             : {failed} ❌")
print(f"Overall status     : {'PASS ✅' if results.success else 'FAIL ❌'}")

Total expectations : 10
Passed             : 10 ✅
Failed             : 0 ❌
Overall status     : PASS ✅


In [13]:
bad_data = df.copy()

# Introduce bad values intentionally
bad_data.loc[0, "age"] = -999
bad_data.loc[1, "MonthlyIncome"] = None
bad_data.loc[2, "SeriousDlqin2yrs"] = 5

print("Created bad data with 3 intentional errors:")
print(f"  Row 0: age = -999")
print(f"  Row 1: MonthlyIncome = None")
print(f"  Row 2: SeriousDlqin2yrs = 5")

bad_results = validation_definition.run(
    batch_parameters={"dataframe": bad_data}
)

passed_bad = sum(1 for r in bad_results.results if r.success)
failed_bad = sum(1 for r in bad_results.results if not r.success)

print(f"\nValidation on BAD data:")
print(f"  Passed: {passed_bad} ✅")
print(f"  Failed: {failed_bad} ❌")
print(f"  Status: {'PASS' if bad_results.success else 'FAIL ❌ — GE caught the errors!'}")

Created bad data with 3 intentional errors:
  Row 0: age = -999
  Row 1: MonthlyIncome = None
  Row 2: SeriousDlqin2yrs = 5


Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]


Validation on BAD data:
  Passed: 7 ✅
  Failed: 3 ❌
  Status: FAIL ❌ — GE caught the errors!


In [14]:
import json
from datetime import datetime

summary = {
    "run_date"           : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset"            : "loan_data_clean.csv",
    "total_rows"         : len(df),
    "total_expectations" : len(results.results),
    "passed"             : passed,
    "failed"             : failed,
    "overall_status"     : "PASS" if results.success else "FAIL"
}

with open("../outputs/ge_validation_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to outputs/ge_validation_results.json")
print("\nDay 15 complete!")

Saved to outputs/ge_validation_results.json

Day 15 complete!
